## Motivation:
Infer # of residents per housing type in census:
Type: SFH, MFH, Low-rise, Hi-rise
Age


# Data Sources:
census_population.csv - total_pop
census_year_built.csv - 10-year increments between 1939 and 2019 
census_housing.csv - Num units

## First Idea:
Just do something simple. Since housing age isn't recorded with housing type, just examine housing age + population density in a covariate analysis. 

In [1]:
import pandas as pd

# df_pop = pd.read_csv("../raw_data/census_housing.csv")
# df_yb = pd.read_csv("../raw_data/census_year_built.csv")

# Clean incidents data (only structure fires)
raw_incidents = pd.read_csv("../processed_data/incidents_clean.csv")

# relevant_incidents = ['BOX -Structure Fire','BOXL- Structure Fire','BOXHI - HIRISE Structure Fire','BOXMID -MIDRISE Structure Fire']
incidents_df = raw_incidents[raw_incidents['incident_category'] == 'Structure Fire']

incidents_df = incidents_df[['incident_number', 'calendaryear', 'month', 'incdate',
       'problem', 'responsearea', 'prioritydescription',
       'council_district', 'location', 'longitude', 'latitude']]

In [6]:
incidents_gdf = gpd.GeoDataFrame(
    incidents_df, 
    geometry=gpd.points_from_xy(incidents_df['longitude'], incidents_df['latitude']),
    crs='EPSG:4326'
)

In [5]:
import geopandas as gpd
tracts = gpd.read_file('../raw_data/tl_2023_48_tract.shp')
tracts = tracts[tracts['COUNTYFP'].isin(['453','209','491'])]
tracts = tracts.to_crs(epsg=4326)

In [14]:
incidents_with_tract_within = gpd.sjoin(
    incidents_gdf,
    tracts,
    how='inner',
    predicate='within'
)
incidents_with_tract_within.shape

(3102, 26)

In [13]:
incidents_with_tract_intersects = gpd.sjoin(
    incidents_gdf,
    tracts,
    how='inner',
    predicate='intersects'
)
incidents_with_tract_intersects.shape

(3255, 26)

In [16]:
incidents_with_tract_intersects[['incident_number','latitude','longitude','NAME']]

,incident_number,latitude,longitude,NAME
5,22000037,30.364969,-97.692635,410
22,22000085,30.316076,-97.714795,15.03
28,22000122,30.372331,-97.612957,449
35,22000252,30.213776,-97.765376,24.03
47,22000440,30.359055,-97.691949,431
...,...,...,...,...
20904,24182205,30.237659,-97.727587,23.16
20913,24182458,30.179791,-97.821621,318
20917,24182496,30.274990,-97.762534,12
20918,24182498,30.274920,-97.762579,12


In [17]:
incidents_with_tract_within[['incident_number','latitude','longitude','NAME']]

,incident_number,latitude,longitude,NAME
5,22000037,30.364969,-97.692635,410
22,22000085,30.316076,-97.714795,15.03
28,22000122,30.372331,-97.612957,449
35,22000252,30.213776,-97.765376,24.03
47,22000440,30.359055,-97.691949,431
...,...,...,...,...
20904,24182205,30.237659,-97.727587,23.16
20913,24182458,30.179791,-97.821621,318
20917,24182496,30.274990,-97.762534,12
20918,24182498,30.274920,-97.762579,12


In [19]:
incidents_with_tract_intersects[
    incidents_with_tract_intersects['incident_number']
    .isin(incidents_with_tract_within['incident_number'])
    ][['incident_number','latitude','longitude','NAME']]

,incident_number,latitude,longitude,NAME
5,22000037,30.364969,-97.692635,410
22,22000085,30.316076,-97.714795,15.03
28,22000122,30.372331,-97.612957,449
35,22000252,30.213776,-97.765376,24.03
47,22000440,30.359055,-97.691949,431
...,...,...,...,...
20904,24182205,30.237659,-97.727587,23.16
20913,24182458,30.179791,-97.821621,318
20917,24182496,30.274990,-97.762534,12
20918,24182498,30.274920,-97.762579,12


In [20]:
prod_df = pd.read_csv("../processed_data/sf_incidents_with_tracts.csv")
prod_df[prod_df['incident_number'] == '22000037']

,incident_number,calendaryear,month,incdate,call_type,problem,responsearea,jurisdiction,prioritydescription,council_district,location,longitude,latitude,census_tracts


# Conclusion from Incident -> Tract Overlay:
Lots of Fire Incident Lat/Longs are recorded on the boundaries of census tracts (since census tract boundaries follow major roads).

### Solution:
We could get better demographic inference by just averaging the demographics of neighboring census boundaries.

In [ ]:
# GIS Summary Data
import geopandas as gpd
# processed_data/response_areas_with_demographics.geojson  # Has sh


def aggregate_incidents_by_response_area(incidents_df):
    """
    Count incidents by response area and category.
    """
    print("\nAggregating incidents by response area...")
    
    # Total incidents
    total_counts = incidents_df.groupby('response_area_id').size().reset_index(name='total_incidents')
    
    # By year (for annualized rates)
    year_col = None
    for col in incidents_df.columns:
        if 'year' in col.lower():
            year_col = col
            break
    
    if year_col:
        years = incidents_df[year_col].nunique()
        total_counts['years_of_data'] = years
        print(f"  Data spans {years} years")
    
    print(f"  Aggregated to {len(total_counts)} response areas")
    
    return total_counts

def merge_incidents_with_demographics(incident_counts_df, response_areas_gdf):
    """
    Merge incident counts with response area demographics.
    """
    print("\nMerging incidents with demographics...")
    
    # Ensure ID columns are same type
    incident_counts_df['response_area_id'] = incident_counts_df['response_area_id'].astype(str)
    response_areas_gdf['response_area_id'] = response_areas_gdf['response_area_id'].astype(str)
    
    # Merge
    merged = response_areas_gdf.merge(incident_counts_df, on='response_area_id', how='left')
    
    # Fill NaN incident counts with 0
    incident_cols = ['total_incidents', 'structure_fires', 'vehicle_fires', 'outdoor_fires']
    for col in incident_cols:
        if col in merged.columns:
            merged[col] = merged[col].fillna(0)
    
    # Calculate per-capita rates (per 1,000 population)
    if 'population' in merged.columns:
        merged['incidents_per_1000_pop'] = np.where(
            merged['population'] > 0,
            (merged['total_incidents'] / merged['population']) * 1000,
            np.nan
        )
    
    # Calculate per-housing-unit rates (per 1,000 units)
    if 'total_units' in merged.columns:
        merged['incidents_per_1000_units'] = np.where(
            merged['total_units'] > 0,
            (merged['total_incidents'] / merged['total_units']) * 1000,
            np.nan
        )
        
    # Annualize if we have years
    if 'years_of_data' in merged.columns:
        years = merged['years_of_data'].iloc[0]
        merged['annual_incidents_per_1000_pop'] = merged['incidents_per_1000_pop'] / years
        merged['annual_incidents_per_1000_units'] = merged['incidents_per_1000_units'] / years
    
    print(f"  Merged dataset: {len(merged)} response areas")
    
    return merged


In [ ]:
for idx, incident in incidents_gdf.iterrows():
    point = incident.geometry
    
    # Find all tracts that touch this point (within or on boundary)
    touching_tracts = tracts[tracts.geometry.touches(point) | tracts.geometry.contains(point)]
    
    if len(touching_tracts) > 0:
        tract_list = touching_tracts['Census Tract'].tolist()
        if len(tract_list) > 1:
            boundary_count += 1
    else:
        tract_list = [None]
    
    tract_arrays.append(tract_list)

# Checking Census Income Data

In [5]:
import os
import requests

def download_census_api(table, variables, filename, description, group=False):
    if os.path.isfile(filename):
        print("File already exists, skipping download.")
        return True
    
    """Download data from Census API"""
    print(f"\n{'='*60}")
    print(f"Downloading: {description}")
    print(f"Table: {table}")
    print('='*60)
    
    base_url = "https://api.census.gov/data/2022/acs/acs5"
    if group:
        base_url += "/subject"
    var_string = ",".join(variables)
    county_string = ",".join(AUSTIN_COUNTIES.values())

    url = f"{base_url}?get={var_string}&for=tract:*&in=state:48&in=county:{county_string}"
    print(f"URL: {url}")

    try:
        response = requests.get(url, timeout=60)
        response.raise_for_status()
        
        data = response.json()

        # Convert to CSV format with human-readable column labels
        header = data[0]
        header_renamed = [ALL_CENSUS_VARS.get(col, col) for col in header]

        import csv
        with open(filename, 'w', newline='') as f:
            writer = csv.writer(f)
            writer.writerow(header_renamed)
            for row in data[1:]:
                writer.writerow(row)

        print(f"✓ Downloaded {len(data)-1} records")
        return True
        
    except Exception as e:
        print(f"✗ Failed: {e}")
        return False
    
# Census variable definitions for ATX fire resource analysis

# Counties
AUSTIN_COUNTIES = {
    "TRAVIS": '453', 
    "HAYS": '209', 
    "WILLIAMSON": '491',
    }

# B25034: Year structure built
YEAR_BUILT_VARS = {
    "B25034_001E": "Total",
    "B25034_002E": "Built 2020 or later",
    "B25034_003E": "Built 2010-2019",
    "B25034_004E": "Built 2000-2009",
    "B25034_005E": "Built 1990-1999",
    "B25034_006E": "Built 1980-1989",
    "B25034_007E": "Built 1970-1979",
    "B25034_008E": "Built 1960-1969",
    "B25034_009E": "Built 1950-1959",
    "B25034_010E": "Built 1940-1949",
    "B25034_011E": "Built 1939 or earlier",
    "NAME": "Census tract name"
}

# B25024: Housing units by type
HOUSING_VARS = {
    "B25024_001E": "Total",
    "B25024_002E": "1-unit, detached",
    "B25024_003E": "1-unit, attached",
    "B25024_004E": "2 units",
    "B25024_005E": "3-4 units",
    "B25024_006E": "5-9 units",
    "B25024_007E": "10-19 units",
    "B25024_008E": "20-49 units",
    "B25024_009E": "50 or more units",
    "B25024_010E": "Mobile home",
    "B25024_011E": "Boat, RV, van",
    "NAME": "Census tract name"
}

# B01003: Total population
POPULATION_VARS = {
    "B01003_001E": "Total population",
    "NAME": "Census tract name"
}

INCOME_VARS = {
    "S1901_C01_001E": "Estimate!!Households!!Total",
    "S1901_C01_001M": "Margin of Error!!Households!!Total",
    "S1901_C01_002E": "Estimate!!Households!!Total!!Less than $10,000",
    "S1901_C01_002M": "Margin of Error!!Households!!Total!!Less than $10,000",
    "S1901_C01_003E": "Estimate!!Households!!Total!!$10,000 to $14,999",
    "S1901_C01_003M": "Margin of Error!!Households!!Total!!$10,000 to $14,999",
    "S1901_C01_004E": "Estimate!!Households!!Total!!$15,000 to $24,999",
    "S1901_C01_004M": "Margin of Error!!Households!!Total!!$15,000 to $24,999",
    "S1901_C01_005E": "Estimate!!Households!!Total!!$25,000 to $34,999",
    "S1901_C01_005M": "Margin of Error!!Households!!Total!!$25,000 to $34,999",
    "S1901_C01_006E": "Estimate!!Households!!Total!!$35,000 to $49,999",
    "S1901_C01_006M": "Margin of Error!!Households!!Total!!$35,000 to $49,999",
    "S1901_C01_007E": "Estimate!!Households!!Total!!$50,000 to $74,999",
    "S1901_C01_007M": "Margin of Error!!Households!!Total!!$50,000 to $74,999",
    "S1901_C01_008E": "Estimate!!Households!!Total!!$75,000 to $99,999",
    "S1901_C01_008M": "Margin of Error!!Households!!Total!!$75,000 to $99,999",
    "S1901_C01_009E": "Estimate!!Households!!Total!!$100,000 to $149,999",
    "S1901_C01_009M": "Margin of Error!!Households!!Total!!$100,000 to $149,999",
    "S1901_C01_010E": "Estimate!!Households!!Total!!$150,000 to $199,999",
    "S1901_C01_010M": "Margin of Error!!Households!!Total!!$150,000 to $199,999",
    "S1901_C01_011E": "Estimate!!Households!!Total!!$200,000 or more",
    "S1901_C01_011M": "Margin of Error!!Households!!Total!!$200,000 or more",
    "S1901_C01_012E": "Estimate!!Households!!Median income (dollars)",
    "S1901_C01_012M": "Margin of Error!!Households!!Median income (dollars)",
    "S1901_C01_013E": "Estimate!!Households!!Mean income (dollars)",
    "S1901_C01_013M": "Margin of Error!!Households!!Mean income (dollars)",
    "S1901_C01_014E": "Estimate!!Households!!PERCENT ALLOCATED!!Household income in the past 12 months",
    "S1901_C01_014M": "Margin of Error!!Households!!PERCENT ALLOCATED!!Household income in the past 12 months",
    "S1901_C01_015E": "Estimate!!Households!!PERCENT ALLOCATED!!Family income in the past 12 months",
    "S1901_C01_015M": "Margin of Error!!Households!!PERCENT ALLOCATED!!Family income in the past 12 months",
    "S1901_C01_016E": "Estimate!!Households!!PERCENT ALLOCATED!!Nonfamily income in the past 12 months",
    "S1901_C01_016M": "Margin of Error!!Households!!PERCENT ALLOCATED!!Nonfamily income in the past 12 months",
    "S1901_C02_001E": "Estimate!!Families!!Total",
    "S1901_C02_001M": "Margin of Error!!Families!!Total",
    "S1901_C02_002E": "Estimate!!Families!!Total!!Less than $10,000",
    "S1901_C02_002M": "Margin of Error!!Families!!Total!!Less than $10,000",
    "S1901_C02_003E": "Estimate!!Families!!Total!!$10,000 to $14,999",
    "S1901_C02_003M": "Margin of Error!!Families!!Total!!$10,000 to $14,999",
    "S1901_C02_004E": "Estimate!!Families!!Total!!$15,000 to $24,999",
    "S1901_C02_004M": "Margin of Error!!Families!!Total!!$15,000 to $24,999",
    "S1901_C02_005E": "Estimate!!Families!!Total!!$25,000 to $34,999",
    "S1901_C02_005M": "Margin of Error!!Families!!Total!!$25,000 to $34,999",
    "S1901_C02_006E": "Estimate!!Families!!Total!!$35,000 to $49,999",
    "S1901_C02_006M": "Margin of Error!!Families!!Total!!$35,000 to $49,999",
    "S1901_C02_007E": "Estimate!!Families!!Total!!$50,000 to $74,999",
    "S1901_C02_007M": "Margin of Error!!Families!!Total!!$50,000 to $74,999",
    "S1901_C02_008E": "Estimate!!Families!!Total!!$75,000 to $99,999",
    "S1901_C02_008M": "Margin of Error!!Families!!Total!!$75,000 to $99,999",
    "S1901_C02_009E": "Estimate!!Families!!Total!!$100,000 to $149,999",
    "S1901_C02_009M": "Margin of Error!!Families!!Total!!$100,000 to $149,999",
    "S1901_C02_010E": "Estimate!!Families!!Total!!$150,000 to $199,999",
    "S1901_C02_010M": "Margin of Error!!Families!!Total!!$150,000 to $199,999",
    "S1901_C02_011E": "Estimate!!Families!!Total!!$200,000 or more",
    "S1901_C02_011M": "Margin of Error!!Families!!Total!!$200,000 or more",
    "S1901_C02_012E": "Estimate!!Families!!Median income (dollars)",
    "S1901_C02_012M": "Margin of Error!!Families!!Median income (dollars)",
    "S1901_C02_013E": "Estimate!!Families!!Mean income (dollars)",
    "S1901_C02_013M": "Margin of Error!!Families!!Mean income (dollars)",
    "S1901_C02_014E": "Estimate!!Families!!PERCENT ALLOCATED!!Household income in the past 12 months",
    "S1901_C02_014M": "Margin of Error!!Families!!PERCENT ALLOCATED!!Household income in the past 12 months",
    "S1901_C02_015E": "Estimate!!Families!!PERCENT ALLOCATED!!Family income in the past 12 months",
    "S1901_C02_015M": "Margin of Error!!Families!!PERCENT ALLOCATED!!Family income in the past 12 months",
    "S1901_C02_016E": "Estimate!!Families!!PERCENT ALLOCATED!!Nonfamily income in the past 12 months",
    "S1901_C02_016M": "Margin of Error!!Families!!PERCENT ALLOCATED!!Nonfamily income in the past 12 months",
    "S1901_C03_001E": "Estimate!!Married-couple families!!Total",
    "S1901_C03_001M": "Margin of Error!!Married-couple families!!Total",
    "S1901_C03_002E": "Estimate!!Married-couple families!!Total!!Less than $10,000",
    "S1901_C03_002M": "Margin of Error!!Married-couple families!!Total!!Less than $10,000",
    "S1901_C03_003E": "Estimate!!Married-couple families!!Total!!$10,000 to $14,999",
    "S1901_C03_003M": "Margin of Error!!Married-couple families!!Total!!$10,000 to $14,999",
    "S1901_C03_004E": "Estimate!!Married-couple families!!Total!!$15,000 to $24,999",
    "S1901_C03_004M": "Margin of Error!!Married-couple families!!Total!!$15,000 to $24,999",
    "S1901_C03_005E": "Estimate!!Married-couple families!!Total!!$25,000 to $34,999",
    "S1901_C03_005M": "Margin of Error!!Married-couple families!!Total!!$25,000 to $34,999",
    "S1901_C03_006E": "Estimate!!Married-couple families!!Total!!$35,000 to $49,999",
    "S1901_C03_006M": "Margin of Error!!Married-couple families!!Total!!$35,000 to $49,999",
    "S1901_C03_007E": "Estimate!!Married-couple families!!Total!!$50,000 to $74,999",
    "S1901_C03_007M": "Margin of Error!!Married-couple families!!Total!!$50,000 to $74,999",
    "S1901_C03_008E": "Estimate!!Married-couple families!!Total!!$75,000 to $99,999",
    "S1901_C03_008M": "Margin of Error!!Married-couple families!!Total!!$75,000 to $99,999",
    "S1901_C03_009E": "Estimate!!Married-couple families!!Total!!$100,000 to $149,999",
    "S1901_C03_009M": "Margin of Error!!Married-couple families!!Total!!$100,000 to $149,999",
    "S1901_C03_010E": "Estimate!!Married-couple families!!Total!!$150,000 to $199,999",
    "S1901_C03_010M": "Margin of Error!!Married-couple families!!Total!!$150,000 to $199,999",
    "S1901_C03_011E": "Estimate!!Married-couple families!!Total!!$200,000 or more",
    "S1901_C03_011M": "Margin of Error!!Married-couple families!!Total!!$200,000 or more",
    "S1901_C03_012E": "Estimate!!Married-couple families!!Median income (dollars)",
    "S1901_C03_012M": "Margin of Error!!Married-couple families!!Median income (dollars)",
    "S1901_C03_013E": "Estimate!!Married-couple families!!Mean income (dollars)",
    "S1901_C03_013M": "Margin of Error!!Married-couple families!!Mean income (dollars)",
    "S1901_C03_014E": "Estimate!!Married-couple families!!PERCENT ALLOCATED!!Household income in the past 12 months",
    "S1901_C03_014M": "Margin of Error!!Married-couple families!!PERCENT ALLOCATED!!Household income in the past 12 months",
    "S1901_C03_015E": "Estimate!!Married-couple families!!PERCENT ALLOCATED!!Family income in the past 12 months",
    "S1901_C03_015M": "Margin of Error!!Married-couple families!!PERCENT ALLOCATED!!Family income in the past 12 months",
    "S1901_C03_016E": "Estimate!!Married-couple families!!PERCENT ALLOCATED!!Nonfamily income in the past 12 months",
    "S1901_C03_016M": "Margin of Error!!Married-couple families!!PERCENT ALLOCATED!!Nonfamily income in the past 12 months",
    "S1901_C04_001E": "Estimate!!Nonfamily households!!Total",
    "S1901_C04_001M": "Margin of Error!!Nonfamily households!!Total",
    "S1901_C04_002E": "Estimate!!Nonfamily households!!Total!!Less than $10,000",
    "S1901_C04_002M": "Margin of Error!!Nonfamily households!!Total!!Less than $10,000",
    "S1901_C04_003E": "Estimate!!Nonfamily households!!Total!!$10,000 to $14,999",
    "S1901_C04_003M": "Margin of Error!!Nonfamily households!!Total!!$10,000 to $14,999",
    "S1901_C04_004E": "Estimate!!Nonfamily households!!Total!!$15,000 to $24,999",
    "S1901_C04_004M": "Margin of Error!!Nonfamily households!!Total!!$15,000 to $24,999",
    "S1901_C04_005E": "Estimate!!Nonfamily households!!Total!!$25,000 to $34,999",
    "S1901_C04_005M": "Margin of Error!!Nonfamily households!!Total!!$25,000 to $34,999",
    "S1901_C04_006E": "Estimate!!Nonfamily households!!Total!!$35,000 to $49,999",
    "S1901_C04_006M": "Margin of Error!!Nonfamily households!!Total!!$35,000 to $49,999",
    "S1901_C04_007E": "Estimate!!Nonfamily households!!Total!!$50,000 to $74,999",
    "S1901_C04_007M": "Margin of Error!!Nonfamily households!!Total!!$50,000 to $74,999",
    "S1901_C04_008E": "Estimate!!Nonfamily households!!Total!!$75,000 to $99,999",
    "S1901_C04_008M": "Margin of Error!!Nonfamily households!!Total!!$75,000 to $99,999",
    "S1901_C04_009E": "Estimate!!Nonfamily households!!Total!!$100,000 to $149,999",
    "S1901_C04_009M": "Margin of Error!!Nonfamily households!!Total!!$100,000 to $149,999",
    "S1901_C04_010E": "Estimate!!Nonfamily households!!Total!!$150,000 to $199,999",
    "S1901_C04_010M": "Margin of Error!!Nonfamily households!!Total!!$150,000 to $199,999",
    "S1901_C04_011E": "Estimate!!Nonfamily households!!Total!!$200,000 or more",
    "S1901_C04_011M": "Margin of Error!!Nonfamily households!!Total!!$200,000 or more",
    "S1901_C04_012E": "Estimate!!Nonfamily households!!Median income (dollars)",
    "S1901_C04_012M": "Margin of Error!!Nonfamily households!!Median income (dollars)",
    "S1901_C04_013E": "Estimate!!Nonfamily households!!Mean income (dollars)",
    "S1901_C04_013M": "Margin of Error!!Nonfamily households!!Mean income (dollars)",
    "S1901_C04_014E": "Estimate!!Nonfamily households!!PERCENT ALLOCATED!!Household income in the past 12 months",
    "S1901_C04_014M": "Margin of Error!!Nonfamily households!!PERCENT ALLOCATED!!Household income in the past 12 months",
    "S1901_C04_015E": "Estimate!!Nonfamily households!!PERCENT ALLOCATED!!Family income in the past 12 months",
    "S1901_C04_015M": "Margin of Error!!Nonfamily households!!PERCENT ALLOCATED!!Family income in the past 12 months",
    "S1901_C04_016E": "Estimate!!Nonfamily households!!PERCENT ALLOCATED!!Nonfamily income in the past 12 months",
    "S1901_C04_016M": "Margin of Error!!Nonfamily households!!PERCENT ALLOCATED!!Nonfamily income in the past 12 months",
    "NAME": "Census tract name",
}

ALL_CENSUS_VARS = {
    **POPULATION_VARS,
    **HOUSING_VARS,
    **YEAR_BUILT_VARS,
    **INCOME_VARS
}


In [8]:
income_data = download_census_api(
        "S1901",
        ["group(S1901)"],
        "census_income.csv",
        "Median Income by Tract",
        group=True
    )


Downloading: Median Income by Tract
Table: S1901
URL: https://api.census.gov/data/2022/acs/acs5/subject?get=group(S1901)&for=tract:*&in=state:48&in=county:453,209,491
✓ Downloaded 471 records


In [ ]:
# good
https://api.census.gov/data/2022/acs/acs5/subject?get=group(S1901)&for=tract:*&in=state:48&in=county:453,209,491
# bad
https://api.census.gov/data/2022/acs/acs5?get=group(S1901)&for=tract:*&in=state:48&in=county:453,209,491    

In [8]:
housing_vars = ["B25024_001E"]  # Total
housing_vars += [f"B25024_{str(i).zfill(3)}E" for i in range(2, 12)]  # Breakdown
housing_vars += ["NAME"]

census_housing = download_census_api(
    "B25024",
    housing_vars,
    "raw_data/census_housing.csv",
    "Census Housing Units by Type (Travis County)"
)    


Downloading: Census Housing Units by Type (Travis County)
Table: B25024
URL: https://api.census.gov/data/2022/acs/acs5?get=B25024_001E,B25024_002E,B25024_003E,B25024_004E,B25024_005E,B25024_006E,B25024_007E,B25024_008E,B25024_009E,B25024_010E,B25024_011E,NAME&for=tract:*&in=state:48&in=county:453,209,491
✗ Failed: [Errno 2] No such file or directory: 'raw_data/census_housing.csv'


In [10]:
import pandas as pd
income = pd.read_csv("../raw_data/census_income.csv")

In [12]:
income.columns

Index(['GEO_ID', 'Census tract name', 'Estimate!!Households!!Total',
       'S1901_C01_001EA', 'Margin of Error!!Households!!Total',
       'S1901_C01_001MA', 'Estimate!!Households!!Total!!Less than $10,000',
       'S1901_C01_002EA',
       'Margin of Error!!Households!!Total!!Less than $10,000',
       'S1901_C01_002MA',
       ...
       'S1901_C04_015EA',
       'Margin of Error!!Nonfamily households!!PERCENT ALLOCATED!!Family income in the past 12 months',
       'S1901_C04_015MA',
       'Estimate!!Nonfamily households!!PERCENT ALLOCATED!!Nonfamily income in the past 12 months',
       'S1901_C04_016EA',
       'Margin of Error!!Nonfamily households!!PERCENT ALLOCATED!!Nonfamily income in the past 12 months',
       'S1901_C04_016MA', 'state', 'county', 'tract'],
      dtype='str', length=261)

In [ ]:
income[['GEO_ID','Census tract name',"S1901_C01_012EA","Estimate!!Households!!Median income (dollars)"]].head()

,GEO_ID,Census tract name,S1901_C01_012EA,Estimate!!Households!!Median income (dollars)
0,1400000US48209010100,Census Tract 101; Hays County; Texas,NaN,41521
1,1400000US48209010200,Census Tract 102; Hays County; Texas,NaN,33409
2,1400000US48209010302,Census Tract 103.02; Hays County; Texas,NaN,45036
3,1400000US48209010305,Census Tract 103.05; Hays County; Texas,NaN,47164
4,1400000US48209010306,Census Tract 103.06; Hays County; Texas,NaN,50800
